### Context

This demo implements a CEOS-style scheme for harmonised identifiers across:
- **sensor** (specific observing device),
- **instrument** (type/class of sensors),
- **platform**, **constellation**, **mission**, **campaign**.

Each field has 4 “incarnations” for different use cases:
- `fullName` (≤256), `shortName` (≤32), `acronym` (≤16), `mnemonic` (≤8, file/URI safe).

Design notes:
- No colons (`:`) in IDs to remain filename-safe (use `-` and `_`).
- Distinguish **sensor** vs **instrument** (sensor = concrete device; instrument = class/type).
- Support **arrays** for fused products (multiple instruments/sensors).
- Provide **STAC/EO mappings** (platform/mission/constellation + satellite extension).
- Extend to **bands/modes**.


# Imports
Import required libraries

In [1]:
# Bootstrap: ensure required packages are installed (works in Cursor + Colab)
try:
    import pandas as pd  # noqa: F401
except Exception:
    %pip install -q pandas
    import pandas as pd

import re
from typing import Dict, List


In [2]:
# ---------- Import local idkit library ----------

import os, sys, pathlib

# Add this notebook’s directory to sys.path so Python can find idkit.py
# (In Colab, Cursor, or local runs, this ensures we can import our custom module
# without having to package/install it. Keeps everything self-contained.)
nb_dir = pathlib.Path().resolve()
if str(nb_dir) not in sys.path:
    sys.path.insert(0, str(nb_dir))

# Import our identifier toolkit.
# - IdKit: the main class that generates canonical IDs for each row
# - IdConfig: configuration (length limits, inclusion of platform codes, etc.)
# - add_band_ids: helper to extend IDs to bands/modes (e.g. MSI_B2, PALSAR_HH)
# - validate_lengths: quick check that IDs respect the length constraints
from idkit import IdKit, IdConfig, add_band_ids, validate_lengths

# Configure and instantiate the IdKit generator
# These limits reflect the proposal in the email thread:
# - shortName ≤32
# - acronym ≤16
# - mnemonic ≤8 (file/URI-safe)
# - flat IDs ≤48 (safe for filenames)
# - hierarchical IDs ≤64 (human-readable components)
# We also choose to include the platform code in mnemonics (so S2A vs S2B differ).
cfg = IdConfig(
    short_max=32,
    acronym_max=16,
    mnemonic_max=8,
    flat_max=48,
    hier_max=64,
    include_platform_in_mnemonic=True,
)

# The IdKit instance (idkit) will be used throughout the notebook
# to mint IDs for simulated CEOS DB rows.
idkit = IdKit(cfg)


# Simulated Data, Identifers, Bands

## Simulated CEOS DB Data (Future API Call)

In [3]:
# -------------------------------------------------------------------
# Simulated CEOS Database records
#
# In production, this dataset would be pulled directly from:
#   - a CEOS DB API endpoint (JSON → DataFrame), OR
#   - a SQL query (e.g. SELECT agency, programme, constellation, mission, ...)
#
# For this demo, we hardcode a minimal representative sample.
# The format mirrors what we expect from the DB so that this
# cell can be swapped out later with a live query.
#
# Note:
# - "programme" = umbrella (e.g. Sentinel, Landsat, ALOS)
# - "constellation" = family of platforms with similar payloads
#   (e.g. Sentinel-1 = radar; Sentinel-2 = optical)
# -------------------------------------------------------------------

rows = [
    # ESA / Sentinel-1 (radar constellation)
    {"agency":"ESA","programme":"Sentinel","constellation":"Sentinel-1","mission":"Sentinel-1A",
     "platform":"Sentinel-1A","platform_code":"A",
     "instrument":"C-SAR","sensor":"C-SAR-A",
     "campaign":"", "cospar":"2014-016A", "nssdca":"2014-016A"},

    {"agency":"ESA","programme":"Sentinel","constellation":"Sentinel-1","mission":"Sentinel-1B",
     "platform":"Sentinel-1B","platform_code":"B",
     "instrument":"C-SAR","sensor":"C-SAR-B",
     "campaign":"", "cospar":"2016-025A", "nssdca":"2016-025A"},

    # ESA / Sentinel-2 (optical constellation)
    {"agency":"ESA","programme":"Sentinel","constellation":"Sentinel-2","mission":"Sentinel-2A",
     "platform":"Sentinel-2A","platform_code":"A",
     "instrument":"MSI","sensor":"MSI-A",
     "campaign":"", "cospar":"2015-028A", "nssdca":"2015-028A"},

    {"agency":"ESA","programme":"Sentinel","constellation":"Sentinel-2","mission":"Sentinel-2B",
     "platform":"Sentinel-2B","platform_code":"B",
     "instrument":"MSI","sensor":"MSI-B",
     "campaign":"", "cospar":"2017-013A", "nssdca":"2017-013A"},

    # USGS / Landsat-8
    {"agency":"USGS","programme":"Landsat","constellation":"Landsat","mission":"Landsat-8",
     "platform":"Landsat-8","platform_code":"8",
     "instrument":"OLI","sensor":"OLI-8",
     "campaign":"", "cospar":"2013-008A", "nssdca":"2013-008A"},

    # JAXA / ALOS-2
    {"agency":"JAXA","programme":"ALOS","constellation":"ALOS-2","mission":"ALOS-2",
     "platform":"ALOS-2","platform_code":"",
     "instrument":"PALSAR-2","sensor":"PALSAR-2",
     "campaign":"", "cospar":"2014-029A", "nssdca":"2014-029A"},
]

df = pd.DataFrame(rows)
df


,agency,programme,constellation,mission,platform,platform_code,instrument,sensor,campaign,cospar,nssdca
0,ESA,Sentinel,Sentinel-1,Sentinel-1A,Sentinel-1A,A,C-SAR,C-SAR-A,,2014-016A,2014-016A
1,ESA,Sentinel,Sentinel-1,Sentinel-1B,Sentinel-1B,B,C-SAR,C-SAR-B,,2016-025A,2016-025A
2,ESA,Sentinel,Sentinel-2,Sentinel-2A,Sentinel-2A,A,MSI,MSI-A,,2015-028A,2015-028A
3,ESA,Sentinel,Sentinel-2,Sentinel-2B,Sentinel-2B,B,MSI,MSI-B,,2017-013A,2017-013A
4,USGS,Landsat,Landsat,Landsat-8,Landsat-8,8,OLI,OLI-8,,2013-008A,2013-008A
5,JAXA,ALOS,ALOS-2,ALOS-2,ALOS-2,,PALSAR-2,PALSAR-2,,2014-029A,2014-029A


## Mint and Display Identifiers

In [4]:
# ---------- Mint identifiers for each row (via idkit lib) ----------

# Generate IDs for the current dataset
df_ids = idkit.mint_dataframe(df)

# Quick invariant checks (length + uniqueness)
validation = validate_lengths(df_ids)
print("Primary ID length constraints:")
display(validation)

# Uniqueness assertions per incarnation (human-friendly fail if something went wrong)
for col in ["shortName", "acronym", "mnemonic", "flat_id", "hierarchical_id"]:
    assert df_ids[col].is_unique, f"{col} not unique"

# Preview with the semantic/context fields surfaced
display_cols = [
    "agency", "constellation", "mission", "campaign",
    "platform", "platform_code", "instrument", "sensor",
    "cospar", "nssdca",
    "fullName", "shortName", "acronym", "mnemonic",
    "flat_id", "hierarchical_id",
]
df_ids[display_cols]

# STAC-aligned projection of key fields (illustrative)
def to_stac_like(df_in: pd.DataFrame) -> pd.DataFrame:
    # Minimal mapping to common STAC-ish fields (not exhaustive)
    out = pd.DataFrame({
        "platform": df_in["platform"],                      # STAC: platform
        "instruments": df_in["instrument"].apply(lambda x: [x]),  # STAC: instruments (array)
        "constellation": df_in["constellation"],            # STAC: constellation
        "mission": df_in["mission"],                        # STAC: mission
        "sat:platform_international_designator": df_in["cospar"], # satellite ext
        # Demo of including our harmonised IDs alongside:
        "id_flat": df_in["flat_id"],
        "id_hier": df_in["hierarchical_id"],
        "id_mnemonic": df_in["mnemonic"],
    })
    return out

stac_like = to_stac_like(df_ids)
stac_like.head(8)

# Example fused product: multi-instrument on a derived product
fused = pd.DataFrame([{
    "agency":"ESA","constellation":"Sentinel","mission":"S-1/S-2 Fused",
    "platform":"Virtual","platform_code":"",
    "instrument":"C-SAR+MSI","sensor":"C-SAR,S2-MSI",  # multiple concrete sensors used
    "campaign":"", "cospar":"", "nssdca":""
}])

# Mint IDs for the fused record too (uses the same rules)
fused_ids = idkit.mint_dataframe(fused)
fused_ids[["mission","instrument","sensor","shortName","mnemonic","flat_id","hierarchical_id"]]


Primary ID length constraints:


flat_ok     True
hier_ok     True
short_ok    True
acro_ok     True
mnem_ok     True
dtype: bool

,mission,instrument,sensor,shortName,mnemonic,flat_id,hierarchical_id
0,S-1/S-2 Fused,C-SAR+MSI,"C-SAR,S2-MSI",S-1S-2Fused-C-SARMSI,S1SCSA,ESA-S1S2FUSED-CSARMSI,A_ESA-M_S1/S2FUSED-I_CSAR+MSI


In [5]:
# Preview the minted identifiers
display_cols = [
    "agency", "mission", "platform_code", "instrument",
    "fullName", "shortName", "acronym", "mnemonic",
    "flat_id", "hierarchical_id"
]
df_ids[display_cols]


,agency,mission,platform_code,instrument,fullName,shortName,acronym,mnemonic,flat_id,hierarchical_id
0,ESA,Sentinel-1A,A,C-SAR,ESA Sentinel-1A A C-SAR,Sentinel-1A-C-SAR-A,S1_CS_A,SENTCSAA,ESA-SENT1A-CSAR-A,A_ESA-M_SENT1A-P_A-I_CSAR
1,ESA,Sentinel-1B,B,C-SAR,ESA Sentinel-1B B C-SAR,Sentinel-1B-C-SAR-B,S1_CS_B,SENTCSAB,ESA-SENT1B-CSAR-B,A_ESA-M_SENT1B-P_B-I_CSAR
2,ESA,Sentinel-2A,A,MSI,ESA Sentinel-2A A MSI,Sentinel-2A-MSI-A,S2_M_A,SENTMSIA,ESA-SENT2A-MSI-A,A_ESA-M_SENT2A-P_A-I_MSI
3,ESA,Sentinel-2B,B,MSI,ESA Sentinel-2B B MSI,Sentinel-2B-MSI-B,S2_M_B,SENTMSIB,ESA-SENT2B-MSI-B,A_ESA-M_SENT2B-P_B-I_MSI
4,USGS,Landsat-8,8,OLI,USGS Landsat-8 8 OLI,Landsat-8-OLI-8,L8_O_8,LANDOLI8,USGS-LANDSAT8-OLI-8,A_USGS-M_LANDSAT8-P_8-I_OLI
5,JAXA,ALOS-2,,PALSAR-2,JAXA ALOS-2 PALSAR-2,ALOS-2-PALSAR-2,A2_P2,ALOSPAL,JAXA-ALOS2-PALSAR2,A_JAXA-M_ALOS2-I_PALSAR2


## Uniqueness Across Platforms (Collision Test)

This example deliberately creates two nearly identical rows 
(same agency, mission, and instrument) that differ only in platform code. 
It demonstrates how the ID scheme avoids collisions and keeps all 
identifiers unique while respecting length constraints.


In [6]:
# Demonstrate uniqueness handling with deliberate collisions using idkit
test = pd.DataFrame([
    {"agency":"TEST", "mission":"Demo-1", "platform_code":"A", "instrument":"XCAM"},
    {"agency":"TEST", "mission":"Demo-1", "platform_code":"B", "instrument":"XCAM"},
])

test_ids = idkit.mint_dataframe(test)
test_ids[
    ["agency","mission","platform_code","instrument",
     "fullName","shortName","acronym","mnemonic","flat_id","hierarchical_id"]
]


,agency,mission,platform_code,instrument,fullName,shortName,acronym,mnemonic,flat_id,hierarchical_id
0,TEST,Demo-1,A,XCAM,TEST Demo-1 A XCAM,Demo-1-XCAM-A,D1_X_A,DEMOXCAA,TEST-DEMO1-XCAM-A,A_TEST-M_DEMO1-P_A-I_XCAM
1,TEST,Demo-1,B,XCAM,TEST Demo-1 B XCAM,Demo-1-XCAM-B,D1_X_B,DEMOXCAB,TEST-DEMO1-XCAM-B,A_TEST-M_DEMO1-P_B-I_XCAM


## Extending Identifiers to Bands and Modes

Many instruments produce multiple spectral bands or polarisation modes.  
To make datasets traceable and file-safe at this finer granularity, 
we extend the canonical IDs with **band-level identifiers**.

This example shows:
- Sentinel-2 MSI and Landsat-8 OLI bands (B01–B08, etc.)  
- Sentinel-1 C-SAR (C-band)  
- ALOS-2 PALSAR-2 polarisations (HH, HV)

The `add_band_ids()` helper appends band/mode tokens to the 
flat and hierarchical IDs, creating unique, length-safe identifiers 
(e.g. `ESA-SENT2-MSI-A-B02` or `A_ESA-M_SENT2-P_A-I_MSI-B_B02`).


In [7]:
# ---------- Bands / modes mapping and expansion ----------

# Minimal illustrative mapping (extend as needed or swap from DB/API later)
INSTRUMENT_BANDS = {
    "MSI": [
        {"band_code":"B01","common":"coastal"},
        {"band_code":"B02","common":"blue"},
        {"band_code":"B03","common":"green"},
        {"band_code":"B04","common":"red"},
        {"band_code":"B08","common":"nir"},
    ],
    "OLI": [
        {"band_code":"B1","common":"coastal"},
        {"band_code":"B2","common":"blue"},
        {"band_code":"B3","common":"green"},
        {"band_code":"B4","common":"red"},
        {"band_code":"B5","common":"nir"},
    ],
    "C-SAR": [
        {"band_code":"CBAND","common":"c_band"},
    ],
    "PALSAR-2": [
        {"band_code":"LBAND","common":"l_band"},
        {"band_code":"HH","common":"hh_pol"},   # illustrative polarisation mode
        {"band_code":"HV","common":"hv_pol"},
    ],
}

# Use the library helper to create band-level IDs from df_ids
df_bands = add_band_ids(df_ids, INSTRUMENT_BANDS)

# Peek at results
df_bands[
    ["agency","mission","platform_code","instrument","band_code","band_common",
     "flat_id_band","hierarchical_id_band"]
].head(20)


,agency,mission,platform_code,instrument,band_code,band_common,flat_id_band,hierarchical_id_band
0,ESA,Sentinel-1A,A,C-SAR,CBAND,c_band,ESA-SENT1A-CSAR-A-CBAND,A_ESA-M_SENT1A-P_A-I_CSAR-B_CBAND
1,ESA,Sentinel-1B,B,C-SAR,CBAND,c_band,ESA-SENT1B-CSAR-B-CBAND,A_ESA-M_SENT1B-P_B-I_CSAR-B_CBAND
2,ESA,Sentinel-2A,A,MSI,B01,coastal,ESA-SENT2A-MSI-A-B01,A_ESA-M_SENT2A-P_A-I_MSI-B_B01
3,ESA,Sentinel-2A,A,MSI,B02,blue,ESA-SENT2A-MSI-A-B02,A_ESA-M_SENT2A-P_A-I_MSI-B_B02
4,ESA,Sentinel-2A,A,MSI,B03,green,ESA-SENT2A-MSI-A-B03,A_ESA-M_SENT2A-P_A-I_MSI-B_B03
5,ESA,Sentinel-2A,A,MSI,B04,red,ESA-SENT2A-MSI-A-B04,A_ESA-M_SENT2A-P_A-I_MSI-B_B04
6,ESA,Sentinel-2A,A,MSI,B08,nir,ESA-SENT2A-MSI-A-B08,A_ESA-M_SENT2A-P_A-I_MSI-B_B08
7,ESA,Sentinel-2B,B,MSI,B01,coastal,ESA-SENT2B-MSI-B-B01,A_ESA-M_SENT2B-P_B-I_MSI-B_B01
8,ESA,Sentinel-2B,B,MSI,B02,blue,ESA-SENT2B-MSI-B-B02,A_ESA-M_SENT2B-P_B-I_MSI-B_B02
9,ESA,Sentinel-2B,B,MSI,B03,green,ESA-SENT2B-MSI-B-B03,A_ESA-M_SENT2B-P_B-I_MSI-B_B03


# Subset Exports

## Lightweight shares (subset exports)

For quick reviews or email shares, export small, focused slices of the registry—
e.g., just **Sentinel-2** (MSI) and **ALOS-2** (PALSAR-2). These JSON files keep
the same schema as the full table but are easy to skim.


In [8]:
# ---------- Subset exports for lightweight sharing (into /export) ----------

# Ensure /export folder exists (relative to the notebook location)
out_dir = pathlib.Path("export")
out_dir.mkdir(parents=True, exist_ok=True)

# Mission-based subsets
s2 = df_ids[df_ids["mission"].str.contains("Sentinel-2", case=False, na=False)]
alos2 = df_ids[df_ids["mission"] == "ALOS-2"]

# Optional: band-level subsets (if df_bands exists)
s2_bands = df_bands[df_bands["mission"].str.contains("Sentinel-2", case=False, na=False)] if "df_bands" in globals() else pd.DataFrame()
alos2_bands = df_bands[df_bands["mission"] == "ALOS-2"] if "df_bands" in globals() else pd.DataFrame()

# Write JSON exports
s2.to_json(out_dir / "sentinel2_registry.json", orient="records", indent=2)
alos2.to_json(out_dir / "alos2_registry.json", orient="records", indent=2)
if not s2_bands.empty:
    s2_bands.to_json(out_dir / "sentinel2_registry_bands.json", orient="records", indent=2)
if not alos2_bands.empty:
    alos2_bands.to_json(out_dir / "alos2_registry_bands.json", orient="records", indent=2)

# Write CSV exports
s2.to_csv(out_dir / "sentinel2_registry.csv", index=False)
alos2.to_csv(out_dir / "alos2_registry.csv", index=False)
if not s2_bands.empty:
    s2_bands.to_csv(out_dir / "sentinel2_registry_bands.csv", index=False)
if not alos2_bands.empty:
    alos2_bands.to_csv(out_dir / "alos2_registry_bands.csv", index=False)

print("Wrote subset exports into:", out_dir.resolve())
for f in sorted(out_dir.glob("*")):
    print(" -", f.name)


Wrote subset exports into: /Users/georgedyke/GitHub/ceos-db-toolkit/colab-notebooks/canonical_id_demo/export
 - alos2_registry.csv
 - alos2_registry.json
 - alos2_registry_bands.csv
 - alos2_registry_bands.json
 - sentinel2_registry.csv
 - sentinel2_registry.json
 - sentinel2_registry_bands.csv
 - sentinel2_registry_bands.json


### How this demo reflects the email thread

- **Disambiguation**: separates *sensor* (concrete device) from *instrument* (type), plus platform/constellation/mission/campaign.
- **Incarnations**: provides `fullName`, `shortName`, `acronym`, `mnemonic`, and file/URI-safe IDs (`flat_id`, `hierarchical_id`).
- **No colons**: only `-` and `_`, per filename constraints.
- **Arrays & fused products**: examples show multiple instruments/sensors in a single record.
- **STAC alignment**: `platform`, `instruments` (array), `constellation`, `mission`, and `sat:platform_international_designator` are mapped explicitly.
- **Bands/modes**: instrument sub-elements are handled in a general way (bands or SAR polarisations).
- **Next steps**: agree governance and persistence (CEOS DB as “bookkeeper”); publish via API + schema.
